# Notebook 09 — Cross-Testbed Transfer (SWaT ⇄ WADI), Corrected Core

The one experiment where MAML has its strongest theoretical case: **real distribution shift**
between different plants. We meta-train on one testbed and few-shot adapt to the other.

**Feature bridge.** SWaT (51) and WADI (123) have no corresponding sensors, so each plant is
projected to a common **32-dim PCA space** (fit on that plant's normal data). This retains
~100% of SWaT and ~97% of WADI variance. Honest limitation: PCA axes are not semantically
aligned across plants, so what can transfer is alignment-free temporal-reconstruction
structure, not sensor correspondence.

**Comparison per direction (src→tgt), all scored on the target's attack windows:**
- **MAML-transfer**  — meta-trained on src, few-shot adapted on tgt.
- **Static-transfer** — conventional AE trained on src, few-shot adapted on tgt.
- **Target-static**  — AE trained directly on tgt (the "just train on target" reference).
- **Scratch**        — random init, adapted on tgt shots only.

Key question: does **MAML-transfer beat Static-transfer**? If yes, meta-learning transfers
better than conventional pretraining. If they match (and both trail Target-static), the null
holds even under favourable cross-domain transfer — the strongest form of the negative result.

### ▶ Run: attach BOTH `swat-maml-data` and `wadi-maml-data`. GPU. Save & Run All (Commit).


## 1 — Imports, load both plants, checkpoint discovery

In [1]:
import os, json, pickle, copy, time, warnings
import numpy as np, torch, torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
warnings.filterwarnings('ignore'); torch.manual_seed(42); np.random.seed(42)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print("Device:",DEVICE)
OUTPUT_PATH="/kaggle/working"; WORK=f"{OUTPUT_PATH}/models"
os.makedirs(WORK,exist_ok=True); os.makedirs(f"{OUTPUT_PATH}/results",exist_ok=True)
def _find(name):
    for r,_,f in os.walk("/kaggle/input"):
        if name in f: return os.path.join(r,name)
    return None
def load_plant(p):
    return (np.load(_find(f"{p}_normal.npy")).astype(np.float32),
            np.load(_find(f"{p}_attack.npy")).astype(np.float32),
            np.load(_find(f"{p}_attack_labels.npy")),
            pickle.load(open(_find(f"{p}_tasks.pkl"),"rb")),
            json.load(open(_find(f"{p}_task_splits.json"))))
DATA={p:load_plant(p) for p in ["swat","wadi"]}
for p in DATA: print(p, "normal", DATA[p][0].shape, "attack", DATA[p][1].shape)
def resolve(name):
    best,bs=None,-1
    dirs=[WORK]+[r for r,_,_ in os.walk("/kaggle/input")]
    for d in dirs:
        q=os.path.join(d,name)
        if os.path.exists(q):
            try: s=torch.load(q,map_location="cpu").get("step",0)
            except: s=0
            if s>=bs: bs,best=s,q
    return best,bs


Device: cuda
swat normal (49500, 51) attack (44991, 51)
wadi normal (78457, 123) attack (17280, 123)


## 2 — Project each plant to common 32-dim PCA space (fit on normal)

In [2]:
D=32; W=30; S=10
def make_windows(a): return np.array([a[i:i+W] for i in range(0,len(a)-W+1,S)],dtype=np.float32)
PROJ={}
for p,(Xn,Xa,ya,tasks,splits) in DATA.items():
    pca=PCA(n_components=D,random_state=42).fit(Xn)
    tn=pca.transform(Xn); lo=tn.min(0); hi=tn.max(0); rg=np.where(hi-lo>1e-8,hi-lo,1.0)
    def pr(x2d): return np.clip((pca.transform(x2d)-lo)/rg,0,1).astype(np.float32)
    Pn=pr(Xn); Pa=pr(Xa)
    # project regime task windows
    ptasks={}
    for k,v in tasks.items():
        w=v["normal_windows"]; n,T,F=w.shape
        ptasks[k]=pr(w.reshape(n*T,F)).reshape(n,T,D).astype(np.float32)
    PROJ[p]={"normal":Pn,"attack":Pa,"labels":ya,"tasks":ptasks,"splits":splits,
             "normal_windows":make_windows(Pn),
             "evr":float(pca.explained_variance_ratio_.sum())}
    print(f"{p}: ->{D}d, variance retained {PROJ[p]['evr']:.3f}, normal windows {PROJ[p]['normal_windows'].shape}")


swat: ->32d, variance retained 1.000, normal windows (4948, 30, 32)
wadi: ->32d, variance retained 0.973, normal windows (7843, 30, 32)


## 3 — Model + corrected FOMAML core (input_size = 32)

In [3]:
class Enc(nn.Module):
    def __init__(s,f,h1=64,h2=32,z=16):
        super().__init__(); s.l1=nn.LSTM(f,h1,batch_first=True); s.l2=nn.LSTM(h1,h2,batch_first=True); s.fc=nn.Linear(h2,z)
    def forward(s,x): o,_=s.l1(x); _,(h,_)=s.l2(o); return s.fc(h.squeeze(0))
class Dec(nn.Module):
    def __init__(s,z=16,h1=32,h2=64,f=32,seq=30):
        super().__init__(); s.seq=seq; s.l1=nn.LSTM(z,h1,batch_first=True); s.l2=nn.LSTM(h1,h2,batch_first=True); s.fc=nn.Linear(h2,f)
    def forward(s,z): r=z.unsqueeze(1).repeat(1,s.seq,1); o,_=s.l1(r); o,_=s.l2(o); return s.fc(o)
class LSTMAE(nn.Module):
    def __init__(s,input_size,seq_len=30): super().__init__(); s.encoder=Enc(input_size); s.decoder=Dec(f=input_size,seq=seq_len)
    def forward(s,x): return s.decoder(s.encoder(x))
    def reconstruction_error(s,x): xh=s.forward(x); return torch.mean((x-xh)**2,dim=(1,2))
criterion=nn.MSELoss()
def sample_ep(w, ss=20, qs=20, rng=np.random):
    idx=rng.permutation(len(w)); need=ss+qs
    if len(w)<need: s=w[rng.choice(len(w),ss,replace=True)]; q=w[rng.choice(len(w),qs,replace=True)]
    else: s=w[idx[:ss]]; q=w[idx[ss:need]]
    return torch.tensor(s,dtype=torch.float32).to(DEVICE), torch.tensor(q,dtype=torch.float32).to(DEVICE)
def inner_adapt(model, support, lr=0.01, steps=10):
    L=copy.deepcopy(model); L.train(); o=torch.optim.SGD(L.parameters(),lr=lr)
    for _ in range(steps): o.zero_grad(); l=criterion(L(support),support); l.backward(); o.step()
    return L
def outer_step(model, opt, batch_windows, steps=10, ss=20, qs=20, train=True, rng=np.random):
    accum=[None]*len(list(model.parameters())); ml=0.0
    for w in batch_windows:
        sup,qry=sample_ep(w,ss,qs,rng); L=inner_adapt(model,sup,0.01,steps); ql=criterion(L(qry),qry)
        if train:
            g=torch.autograd.grad(ql,L.parameters()); accum=[gi.detach() if a is None else a+gi.detach() for a,gi in zip(accum,g)]
        ml+=ql.item()
    ml/=len(batch_windows)
    if train:
        opt.zero_grad()
        for p,a in zip(model.parameters(),accum): p.grad=a/len(batch_windows)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
    return ml
print("core ready")


core ready


## 4 — Train: MAML per direction + static AE per plant (resume/skip aware)

In [4]:
N_OUTER=5000; VAL_EVERY=500; INNER_STEPS=10; TPB=4; SUP=20; QRY=20
def train_maml(src, tgt):
    name=f"maml_{src}2{tgt}"; CK=f"{WORK}/{name}_ckpt.pt"; BE=f"{WORK}/{name}_best.pt"
    rng=np.random.RandomState(42); model=LSTMAE(D).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    regs=[PROJ[src]["tasks"][k] for k in PROJ[src]["splits"]["meta_train"]]
    vals=[PROJ[src]["tasks"][k] for k in PROJ[src]["splits"]["meta_val"]]
    start=0; best=1e9
    rp,rs=resolve(f"{name}_ckpt.pt")
    if rp and rs>0:
        ck=torch.load(rp,map_location=DEVICE); model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); start=ck['step']; best=ck['best']; print(f"  resume {name} @",start)
    if start>=N_OUTER: print(f"  {name} complete"); return
    t0=time.time()
    for step in range(start+1,N_OUTER+1):
        b=[regs[i] for i in rng.choice(len(regs),min(TPB,len(regs)),replace=False)]
        model.train(); outer_step(model,opt,b,INNER_STEPS,SUP,QRY,True,rng)
        if step%VAL_EVERY==0:
            vl=float(np.mean([outer_step(model,opt,[v],INNER_STEPS,SUP,QRY,False,rng) for v in vals]))
            print(f"  {name} step {step} val {vl:.6f} {time.time()-t0:.0f}s")
            torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'step':step,'best':best},CK)
            if vl<best: best=vl; torch.save({'model_state_dict':model.state_dict(),'step':step},BE)
    print(f"  {name} done best {best:.6f}")

def train_static(plant):
    name=f"static_{plant}"; path=_find(f"{name}.pt") or f"{WORK}/{name}.pt"
    m=LSTMAE(D).to(DEVICE)
    if _find(f"{name}.pt"): m.load_state_dict(torch.load(_find(f"{name}.pt"),map_location=DEVICE)['model_state_dict']); print(f"  loaded {name}"); return m
    data=torch.tensor(PROJ[plant]["normal_windows"]); p=torch.randperm(len(data)); cut=int(0.9*len(data))
    tr=data[p[:cut]]; va=data[p[cut:]].to(DEVICE); o=torch.optim.Adam(m.parameters(),lr=1e-3); best=1e9; bad=0; bs=None
    for ep in range(120):
        m.train(); pi=torch.randperm(len(tr))
        for st in range(0,len(tr),128):
            b=tr[pi[st:st+128]].to(DEVICE); o.zero_grad(); l=criterion(m(b),b); l.backward(); o.step()
        m.eval()
        with torch.no_grad(): vl=criterion(m(va),va).item()
        if vl<best-1e-6: best=vl; bad=0; bs={k:v.clone() for k,v in m.state_dict().items()}
        else:
            bad+=1
            if bad>=12: break
    m.load_state_dict(bs); torch.save({'model_state_dict':m.state_dict()},f"{WORK}/{name}.pt"); print(f"  {name} val {best:.6f}"); return m

print("== training MAML both directions =="); train_maml("swat","wadi"); train_maml("wadi","swat")
print("== training static per plant =="); STATIC={p:train_static(p) for p in ["swat","wadi"]}


== training MAML both directions ==
  maml_swat2wadi step 500 val 0.014023 139s
  maml_swat2wadi step 1000 val 0.016357 277s
  maml_swat2wadi step 1500 val 0.015838 414s
  maml_swat2wadi step 2000 val 0.016813 552s
  maml_swat2wadi step 2500 val 0.013818 690s
  maml_swat2wadi step 3000 val 0.014711 828s
  maml_swat2wadi step 3500 val 0.013923 966s
  maml_swat2wadi step 4000 val 0.012806 1104s
  maml_swat2wadi step 4500 val 0.016022 1242s
  maml_swat2wadi step 5000 val 0.014919 1380s
  maml_swat2wadi done best 0.012806
  maml_wadi2swat step 500 val 0.015157 139s
  maml_wadi2swat step 1000 val 0.015550 277s
  maml_wadi2swat step 1500 val 0.013006 416s
  maml_wadi2swat step 2000 val 0.011951 554s
  maml_wadi2swat step 2500 val 0.012511 692s
  maml_wadi2swat step 3000 val 0.008450 830s
  maml_wadi2swat step 3500 val 0.006615 968s
  maml_wadi2swat step 4000 val 0.005963 1107s
  maml_wadi2swat step 4500 val 0.006598 1249s
  maml_wadi2swat step 5000 val 0.006175 1390s
  maml_wadi2swat done be

## 5 — Evaluation (windowed, on target attack) + run both directions

In [5]:
def metrics(scores, yw):
    auc=roc_auc_score(yw,scores); ap=average_precision_score(yw,scores)
    p,r,_=precision_recall_curve(yw,scores); f1=2*p*r/(p+r+1e-12); bi=np.nanargmax(f1)
    return {'roc_auc':round(float(auc),4),'best_f1':round(float(f1[bi]),4),'pr_auc':round(float(ap),4),
            'separation':round(float(scores[yw==1].mean()/scores[yw==0].mean()),3)}
def attack_windows(tgt):
    Xa=PROJ[tgt]["attack"]; ya=PROJ[tgt]["labels"]
    idx=[(i,i+W) for i in range(0,len(Xa)-W+1,S)]
    Wa=np.array([Xa[a:b] for a,b in idx],dtype=np.float32); yw=np.array([int(ya[a:b].any()) for a,b in idx])
    return torch.tensor(Wa).to(DEVICE), yw
def score_adapt(model, tgt, support_size, seed, adapt_steps=10):
    r=np.random.RandomState(seed); nw=PROJ[tgt]["normal_windows"]
    sup=torch.tensor(nw[r.choice(len(nw),support_size,replace=False)],dtype=torch.float32).to(DEVICE)
    L=inner_adapt(model,sup,0.01,adapt_steps); L.eval()
    Wa,yw=attack_windows(tgt)
    with torch.no_grad(): s=L.reconstruction_error(Wa).cpu().numpy()
    return metrics(s,yw)
def score_plain(model, tgt):
    model.eval(); Wa,yw=attack_windows(tgt)
    with torch.no_grad(): s=model.reconstruction_error(Wa).cpu().numpy()
    return metrics(s,yw)

SUPPORT=[20,50,100]; SEEDS=[42,123,456]
def evaluate_direction(src,tgt):
    maml=LSTMAE(D).to(DEVICE); maml.load_state_dict(torch.load(resolve(f"maml_{src}2{tgt}_best.pt")[0],map_location=DEVICE)['model_state_dict'])
    out={}
    out['Target-static']=score_plain(STATIC[tgt],tgt)     # skyline: trained on target
    def avg(modelfn):
        acc={}
        for ss in SUPPORT:
            runs=[modelfn(ss,s) for s in SEEDS]
            acc[ss]={k:round(float(np.mean([d[k] for d in runs])),4) for k in runs[0]}
        return acc
    out['MAML-transfer']=avg(lambda ss,s: score_adapt(maml,tgt,ss,s))
    out['Static-transfer']=avg(lambda ss,s: score_adapt(STATIC[src],tgt,ss,s))
    out['Scratch']=avg(lambda ss,s: score_adapt(LSTMAE(D).to(DEVICE),tgt,ss,s))
    return out
RESULTS={f"{a}->{b}":evaluate_direction(a,b) for a,b in [("swat","wadi"),("wadi","swat")]}
json.dump(RESULTS,open(f"{OUTPUT_PATH}/results/transfer_results.json","w"),indent=2)
print("evaluation complete")


evaluation complete


## 6 — Results: does meta-learning transfer better than conventional pretraining?

In [6]:
for direction,res in RESULTS.items():
    print("="*64); print("DIRECTION:",direction); print("="*64)
    print(f"Target-static (train-on-target reference): ROC-AUC {res['Target-static']['roc_auc']:.4f}  F1 {res['Target-static']['best_f1']:.4f}")
    print(f"\n{'method':>16s} {'shot':>5s} | {'ROC-AUC':>8s} | {'best-F1':>8s} | {'separation':>10s}")
    for m in ['MAML-transfer','Static-transfer','Scratch']:
        for ss in SUPPORT:
            r=res[m][ss]; print(f"{m:>16s} {ss:>5d} | {r['roc_auc']:>8.4f} | {r['best_f1']:>8.4f} | {r['separation']:>10.3f}")
    print("\n  MAML-transfer minus Static-transfer (ROC-AUC):")
    for ss in SUPPORT:
        d=res['MAML-transfer'][ss]['roc_auc']-res['Static-transfer'][ss]['roc_auc']
        print(f"    {ss:3d}-shot: {d:+.4f}")
print("\nRead: MAML-transfer > Static-transfer => meta-learning transfers better (a positive result).")
print("      MAML-transfer ~ Static-transfer, both < Target-static => null holds even across plants.")


DIRECTION: swat->wadi
Target-static (train-on-target reference): ROC-AUC 0.7074  F1 0.2917

          method  shot |  ROC-AUC |  best-F1 | separation
   MAML-transfer    20 |   0.5807 |   0.1900 |      1.072
   MAML-transfer    50 |   0.5868 |   0.1935 |      1.077
   MAML-transfer   100 |   0.5851 |   0.1897 |      1.075
 Static-transfer    20 |   0.5857 |   0.1819 |      1.072
 Static-transfer    50 |   0.5892 |   0.1874 |      1.073
 Static-transfer   100 |   0.5907 |   0.1883 |      1.073
         Scratch    20 |   0.5039 |   0.1943 |      1.011
         Scratch    50 |   0.4762 |   0.1758 |      1.001
         Scratch   100 |   0.5052 |   0.1894 |      1.010

  MAML-transfer minus Static-transfer (ROC-AUC):
     20-shot: -0.0050
     50-shot: -0.0024
    100-shot: -0.0056
DIRECTION: wadi->swat
Target-static (train-on-target reference): ROC-AUC 0.8116  F1 0.7200

          method  shot |  ROC-AUC |  best-F1 | separation
   MAML-transfer    20 |   0.8000 |   0.6838 |      1.524
   M